# Classification Dataset Collection Widget (ROS Camera Topic)

This notebook collects `free` and `blocked` dataset images for collision avoidance on JetRacer by subscribing to the **ROS Camera Topic** (`/csi_cam_0/image_raw`).

### 1. Setup Environment & ROS Node

In [ ]:
import os
import sys
import cv2
import uuid
import base64
import threading
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Add parent search paths to sys.path
curr = Path.cwd()
for p in [curr, curr.parent, curr.parent.parent]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import rospy
from sensor_msgs.msg import Image as ROSImage

try:
    from jetracer.utils import bgr8_to_jpeg
except ImportError:
    from utils import bgr8_to_jpeg

# Initialize ROS Node
try:
    rospy.init_node('collision_avoidance_data_collection_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

### 2. Dataset Folders & Controls Setup

In [ ]:
blocked_dir = 'dataset/blocked'
free_dir = 'dataset/free'

os.makedirs(free_dir, exist_ok=True)
os.makedirs(blocked_dir, exist_ok=True)

button_layout = widgets.Layout(width='140px', height='45px')
free_button = widgets.Button(description='Add Free', button_style='success', icon='check', layout=button_layout)
blocked_button = widgets.Button(description='Add Blocked', button_style='danger', icon='ban', layout=button_layout)

free_count = widgets.IntText(description='Free Count', value=len(os.listdir(free_dir)), disabled=True, layout=widgets.Layout(width='180px'))
blocked_count = widgets.IntText(description='Blocked Count', value=len(os.listdir(blocked_dir)), disabled=True, layout=widgets.Layout(width='180px'))

status_widget = widgets.HTML(value="<p style='color:green;'><b>Ready to collect collision avoidance samples.</b></p>")

camera_html_widget = widgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=widgets.Layout(width='240px', height='240px')
)
snapshot_widget = widgets.Image(format='jpeg', width=224, height=224, layout=widgets.Layout(border='2px solid #00ff00', border_radius='4px'))

### 3. ROS Subscriber & Interactive Capture UI

In [ ]:
latest_ros_image = None
_lock = threading.Lock()

if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

def ros_image_to_cv2(msg):
    im = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
    if msg.encoding in ['rgb8', 'rgb8']:
        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    elif msg.encoding == 'rgba8':
        im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
    elif msg.encoding == 'bgra8':
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    if im.shape[0] != 224 or im.shape[1] != 224:
        im = cv2.resize(im, (224, 224))
    return im

def camera_callback(msg):
    global latest_ros_image
    if not _lock.acquire(blocking=False):
        return
    try:
        latest_ros_image = ros_image_to_cv2(msg)
        jpeg_bytes = bgr8_to_jpeg(latest_ros_image)
        b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
        
        html_str = f'''
        <div style="font-family: monospace; background: #1e1e1e; padding: 6px; border-radius: 6px; display: inline-block;">
            <h5 style="margin:0 0 4px 0; color: #ffffff;">Collision Avoidance ROS Feed</h5>
            <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px;" />
        </div>
        '''
        camera_html_widget.value = html_str
    except Exception:
        pass
    finally:
        try:
            _lock.release()
        except RuntimeError:
            pass

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, camera_callback, queue_size=1, buff_size=2**24)

def save_snapshot(directory):
    global latest_ros_image
    with _lock:
        if latest_ros_image is None:
            status_widget.value = "<p style='color:red;'><b>[!] No ROS camera frame available!</b></p>"
            return False
        img_to_save = latest_ros_image.copy()

    image_path = os.path.join(directory, str(uuid.uuid1())[:8] + '.jpg')
    cv2.imwrite(image_path, img_to_save)
    snapshot_widget.value = bgr8_to_jpeg(img_to_save)
    return True

def save_free(b):
    if save_snapshot(free_dir):
        free_count.value = len(os.listdir(free_dir))
        status_widget.value = f"<p style='color:green;'><b>[+] Saved FREE sample #{free_count.value}</b></p>"

def save_blocked(b):
    if save_snapshot(blocked_dir):
        blocked_count.value = len(os.listdir(blocked_dir))
        status_widget.value = f"<p style='color:red;'><b>[+] Saved BLOCKED sample #{blocked_count.value}</b></p>"

free_button.on_click(save_free)
blocked_button.on_click(save_blocked)

ui_layout = widgets.VBox([
    widgets.HBox([camera_html_widget, snapshot_widget]),
    widgets.HBox([free_count, free_button]),
    widgets.HBox([blocked_count, blocked_button]),
    status_widget
])

display(ui_layout)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")